In [ ]:
import torchvision
import torch.nn as nn
import time
from torch.utils.data import DataLoader
import torch
import gc
import torchvision.transforms as standard_transforms
import numpy as np
import multiprocessing
import os
from PIL import Image
from torch.utils import data

In [ ]:
train_dataset =torchvision.datasets.VOCSegmentation(root='./data',year='2012',download=True,image_set='train')
val_dataset = torchvision.datasets.VOCSegmentation(root='./data',year='2012',download=True,image_set='val')
test_dataset = torchvision.datasets.VOCSegmentation(root='./data',year='2012',download=True,image_set='val')

In [ ]:
class FCN(nn.Module):

    def __init__(self, n_class):
        # TODO: Skeleton code given for default FCN network. Fill in the blanks with the shapes
        super().__init__()
        self.n_class = n_class
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, stride=2, padding=1, dilation=1)
        self.bnd1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1, dilation=1)
        self.bnd2 = nn.BatchNorm2d(64)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1, dilation=1)
        self.bnd3 = nn.BatchNorm2d(128)
        self.conv4 = nn.Conv2d(128, 256, kernel_size=3, stride=2, padding=1, dilation=1)
        self.bnd4 = nn.BatchNorm2d(256)
        self.conv5 = nn.Conv2d(256, 512, kernel_size=3, stride=2, padding=1, dilation=1)
        self.bnd5 = nn.BatchNorm2d(512)
        self.relu = nn.ReLU(inplace=True)
        self.deconv1 = nn.ConvTranspose2d(512, 256, kernel_size=3, stride=2, padding=1, dilation=1, output_padding=1)
        self.bn1 = nn.BatchNorm2d(256)
        self.deconv2 = nn.ConvTranspose2d(256, 128, kernel_size=3, stride=2, padding=1, dilation=1, output_padding=1)
        self.bn2 = nn.BatchNorm2d(128)
        self.deconv3 = nn.ConvTranspose2d(128, 64, kernel_size=3, stride=2, padding=1, dilation=1, output_padding=1)
        self.bn3 = nn.BatchNorm2d(64)
        self.deconv4 = nn.ConvTranspose2d(64, 32, kernel_size=3, stride=2, padding=1, dilation=1, output_padding=1)
        self.bn4 = nn.BatchNorm2d(32)
        self.deconv5 = nn.ConvTranspose2d(32, 16, kernel_size=3, stride=2, padding=1, dilation=1, output_padding=1)
        self.bn5 = nn.BatchNorm2d(16)
        self.classifier = nn.Conv2d(16, self.n_class, kernel_size=1)

    #TODO Complete the forward pass
    def forward(self, x):
        x1 = self.bnd1(self.relu(self.conv1(x)))
        # Complete the forward function for the rest of the encoder
        x2 = self.bnd2(self.relu(self.conv2(x1)))
        x3 = self.bnd3(self.relu(self.conv3(x2)))
        x4 = self.bnd4(self.relu(self.conv4(x3)))
        x5 = self.bnd5(self.relu(self.conv5(x4)))

        y1 = self.bn1(self.relu(self.deconv1(x5)))
        # Complete the forward function for the rest of the decoder
        y2 = self.bn2(self.relu(self.deconv2(y1 + x4)))
        y3 = self.bn3(self.relu(self.deconv3(y2 + x3)))
        y4 = self.bn4(self.relu(self.deconv4(y3 + x2)))
        y5 = self.bn5(self.relu(self.deconv5(y4 + x1)))

        score = self.classifier(y5)

        return score  # size=(N, n_class, H, W)

In [ ]:
def iou(pred, target, n_classes=21):
    """
    Calculate the Intersection over Union (IoU) for predictions.
    Args:
        pred (tensor): Predicted output from the model.
        target (tensor): Ground truth labels.
        n_classes (int, optional): Number of classes. Default is 21.
    Returns:
        float: Mean IoU across all classes.
    """
    ious = []
    pred = torch.argmax(pred, dim=1).view(-1)
    target = target.view(-1)

    for cls in range(n_classes):
        pred_inds = (pred == cls)
        target_inds = (target == cls)

        intersection = (pred_inds & target_inds).sum().item()
        union = (pred_inds | target_inds).sum().item()

        if union == 0:
            continue

        ious.append(intersection / union)

    return float(np.mean(ious)) if ious else 0.0


def pixel_acc(pred, target):
    """
    Calculate pixel-wise accuracy between predictions and targets.
    Args:
        pred (tensor): Predicted output from the model.
        target (tensor): Ground truth labels.
    Returns:
        float: Pixel-wise accuracy.
    """
    pred = torch.argmax(pred, dim=1)
    correct = (pred == target).sum().item()
    total = target.numel()
    return correct / total

In [ ]:
num_classes = 21
ignore_label = 255
root = './data'

'''
color map
0=background, 1=aeroplane, 2=bicycle, 3=bird, 4=boat, 5=bottle # 6=bus, 7=car, 8=cat, 9=chair, 10=cow, 11=diningtable,
12=dog, 13=horse, 14=motorbike, 15=person # 16=potted plant, 17=sheep, 18=sofa, 19=train, 20=tv/monitor
'''


#Feel free to convert this palette to a map
palette = [0, 0, 0, 128, 0, 0, 0, 128, 0, 128, 128, 0, 0, 0, 128, 128, 0, 128, 0, 128, 128,
           128, 128, 128, 64, 0, 0, 192, 0, 0, 64, 128, 0, 192, 128, 0, 64, 0, 128, 192, 0, 128,
           64, 128, 128, 192, 128, 128, 0, 64, 0, 128, 64, 0, 0, 192, 0, 128, 192, 0, 0, 64, 128]  #3 values- R,G,B for every class. First 3 values for class 0, next 3 for
#class 1 and so on......


def make_dataset(mode):
    """
    Creates a list of tuples (image_path, mask_path) for a given dataset mode.

    Asserts that the mode is one of 'train', 'val', or 'test'.
    Based on the mode, it reads corresponding image and mask paths from the VOC dataset.
    For each image, it pairs its path with the corresponding mask path.

    data_list for train is with name train.txt,
    data_list for validation is with name trainval.txt,
    data_list for test is with name val.txt.

    Args:
        mode (str): The mode of the dataset, either 'train', 'val', or 'test'.

    Returns:
        list of tuples: Each tuple contains paths (image_path, mask_path).
    """
    assert mode in ['train', 'val', 'test']
    items = []
    img_path = os.path.join(root, 'VOCdevkit', 'VOC2012', 'JPEGImages')
    mask_path = os.path.join(root, 'VOCdevkit', 'VOC2012', 'SegmentationClass')

    if mode == 'train':
        data_list_file = 'train.txt'
    elif mode == 'val':
        data_list_file = 'val.txt'
    else:
        data_list_file = 'val.txt'

    data_list = [l.strip('\n') for l in open(os.path.join(
        root, 'VOCdevkit', 'VOC2012', 'ImageSets', 'Segmentation', data_list_file)).readlines()]

    for it in data_list:
        item = (os.path.join(img_path, it + '.jpg'), os.path.join(mask_path, it + '.png'))
        items.append(item)

    return items


class VOC(data.Dataset):
    """
    A custom dataset class for VOC dataset.
    Maintain the structure of this so that it is easily compatible with Pytorch's dataloader.

    - Resizes images and masks to a specified width and height.
    - Implements methods to get dataset items and dataset length.

    - TIP: You may add an additional argument for common transformation for both the image and mask
           to help with data augmentation in part 4.c

    Args:
        mode (str): Mode of the dataset ('train', 'val', etc.).
        transform (callable, optional): Transform to be applied to the images.
        target_transform (callable, optional): Transform to be applied to the masks.
    """

    def __init__(self, mode, transform=None, target_transform=None):
        self.imgs = make_dataset(mode)
        if len(self.imgs) == 0:
            raise RuntimeError('Found 0 images, please check the data set')
        self.mode = mode
        self.transform = transform
        self.target_transform = target_transform
        self.width = 256
        self.height = 256

    def __getitem__(self, index):

        img_path, mask_path = self.imgs[index]
        img = Image.open(img_path).convert('RGB').resize((self.width, self.height))
        mask = Image.open(mask_path).resize((self.width, self.height))

        if self.transform is not None:
            img = self.transform(img)
        if self.target_transform is not None:
            mask = self.target_transform(mask)

        mask[mask==ignore_label]=0

        return img, mask

    def __len__(self):
        return len(self.imgs)

In [ ]:
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader

transform = T.Compose([
    T.Resize((256, 256)),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

def mask_transform(mask):
    mask = mask.resize((256, 256))
    return torch.from_numpy(np.array(mask, dtype=np.int64))

In [ ]:
num_workers = multiprocessing.cpu_count()

class MaskToTensor(object):
    def __call__(self, img):
        return torch.from_numpy(np.array(img, dtype=np.int32)).long()

def init_weights(m):
    if isinstance(m, nn.Conv2d) or isinstance(m, nn.ConvTranspose2d):
        torch.nn.init.xavier_uniform_(m.weight.data)
        torch.nn.init.normal_(m.bias.data) #xavier not applicable for biases

def getClassWeights():
    class_counts = torch.zeros(num_classes)
    total_pixels = 0

    for _, labels in train_loader:
        labels = labels.cpu()
        flat_labels = labels[labels != ignore_label]
        if flat_labels.numel() > 0:
            unique_labels, counts = torch.unique(flat_labels, return_counts=True)
            for cls, count in zip(unique_labels, counts):
                class_counts[cls] += count
            total_pixels += flat_labels.numel()

    if total_pixels == 0:
        return torch.ones(num_classes)

    weights = torch.ones(num_classes, dtype=torch.float)
    valid_classes = class_counts > 0
    weights[valid_classes] = total_pixels / (num_classes * class_counts[valid_classes])

    weights = torch.clamp(weights, min=0.5, max=5.0)

    return weights



mean_std = ([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
input_transform = standard_transforms.Compose([
        standard_transforms.ToTensor(),
        standard_transforms.Normalize(*mean_std)
    ])

target_transform = MaskToTensor()


train_dataset = VOC('train', transform=input_transform, target_transform=target_transform)
val_dataset = VOC('val', transform=input_transform, target_transform=target_transform)
test_dataset = VOC('val', transform=input_transform, target_transform=target_transform)

train_loader = DataLoader(dataset=train_dataset, batch_size= 16, shuffle=True, num_workers=num_workers)
val_loader = DataLoader(dataset=val_dataset, batch_size= 16, shuffle=False, num_workers=num_workers)
test_loader = DataLoader(dataset=test_dataset, batch_size= 16, shuffle=False, num_workers=num_workers)


In [ ]:
epochs = 30

n_class = 21

fcn_model = FCN(n_class=n_class)
fcn_model.apply(init_weights)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class_weights = getClassWeights().to(device)

optimizer = torch.optim.SGD(
    fcn_model.parameters(),
    lr=1e-2,
    momentum=0.9,
    weight_decay=1e-4
)

def lr_lambda(epoch):
    if epoch < 3:
        return (epoch + 1) / 3
    return 1.0

warmup_scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
poly_scheduler = torch.optim.lr_scheduler.PolynomialLR(
    optimizer, total_iters=epochs - 3, power=0.9
)

criterion = nn.CrossEntropyLoss(ignore_index=ignore_label, weight=class_weights)

fcn_model = fcn_model.to(device)

In [ ]:
def train(model, epochs):
    """
    Train a deep learning model using mini-batches.

    - Perform forward propagation in each epoch.
    - Compute loss and conduct backpropagation.
    - Update model weights.
    - Evaluate model on validation set for mIoU score.
    - Save model state if mIoU score improves.
    - Implement early stopping if necessary.

    Returns:
        None.
    """
    best_iou_score = 0.0
    for epoch in range(epochs):
        model.train()
        ts = time.time()
        for iter, (inputs, labels) in enumerate(train_loader):
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(inputs), labels)
            loss.backward()
            optimizer.step()
            if iter % 20 == 0:
                print(f"epoch {epoch}, iter {iter}, loss: {loss.item():.4f}")
        if epoch < 3:
            warmup_scheduler.step()
        else:
            poly_scheduler.step()
        print(f"Finish epoch {epoch}, time elapsed {time.time() - ts:.1f}s")
        current_miou = val(model, epoch)
        if current_miou > best_iou_score:
            best_iou_score = current_miou
            torch.save(model.state_dict(), 'best_model.pth')

def val(model, epoch):
    """
    Validate the deep learning model on a validation dataset.

    - Set model to evaluation mode.
    - Disable gradient calculations.
    - Iterate over validation data loader:
        - Perform forward pass to get outputs.
        - Compute loss and accumulate it.
        - Calculate and accumulate mean Intersection over Union (IoU) scores and pixel accuracy.
    - Print average loss, IoU, and pixel accuracy for the epoch.
    - Switch model back to training mode.

    Args:
        epoch (int): The current epoch number.

    Returns:
        tuple: Mean IoU score and mean loss for this validation epoch.
    """
    model.eval()
    losses, iou_scores, accs = [], [], []
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            losses.append(criterion(outputs, labels).item())
            iou_scores.append(iou(outputs, labels))
            accs.append(pixel_acc(outputs, labels))
    print(f"Loss: {np.mean(losses):.4f} | mIoU: {np.mean(iou_scores):.4f} | Acc: {np.mean(accs):.4f}")
    model.train()
    return np.mean(iou_scores)

def modelTest(model):
    """
    Test the deep learning model using a test dataset.

    - Load the model with the best weights.
    - Set the model to evaluation mode.
    - Iterate over the test data loader:
        - Perform forward pass and compute loss.
        - Accumulate loss, IoU scores, and pixel accuracy.
    - Print average loss, IoU, and pixel accuracy for the test data.
    - Switch model back to training mode.

    Returns:
        None. Outputs average test metrics to the console.
    """

    # Load the best model weights
    model.load_state_dict(torch.load('best_model.pth'))
    model.eval()  # Put in eval mode (disables batchnorm/dropout) !
    losses, iou_scores, accs = [], [], []

    with torch.no_grad():  # we don't need to calculate the gradient in the validation/testing

        for iter, (inputs, labels) in enumerate(test_loader):
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            losses.append(criterion(outputs, labels).item())
            iou_scores.append(iou(outputs, labels))
            accs.append(pixel_acc(outputs, labels))

    print(f"Test Loss: {np.mean(losses):.4f} | Test mIoU: {np.mean(iou_scores):.4f} | Test Acc: {np.mean(accs):.4f}")
    model.train()  #TURNING THE TRAIN MODE BACK ON TO ENABLE BATCHNORM/DROPOUT!!

def exportModel(model, inputs):
    """
    Export the output of the model for given inputs.

    - Set the model to evaluation mode.
    - Load the model with the best saved weights.
    - Perform a forward pass with the model to get output.
    - Switch model back to training mode.

    Args:
        inputs: Input data to the model.

    Returns:
        Output from the model for the given inputs.
    """

    model.eval() # Put in eval mode (disables batchnorm/dropout) !

    saved_model_path = "best_model.pth"
    # Then Load your best model using saved_model_path
    model.load_state_dict(torch.load(saved_model_path))

    inputs = inputs.to(device)

    output_image = model(inputs)

    model.train()  #TURNING THE TRAIN MODE BACK ON TO ENABLE BATCHNORM/DROPOUT!!

    return output_image


In [ ]:
val(fcn_model, 0)
train(fcn_model, epochs)
modelTest(fcn_model)

Loss: 3.9838 | mIoU: 0.0007 | Acc: 0.0078
epoch 0, iter 0, loss: 3.7509
epoch 0, iter 20, loss: 3.9620
epoch 0, iter 40, loss: 3.5153
epoch 0, iter 60, loss: 3.2425
epoch 0, iter 80, loss: 3.0177
Finish epoch 0, time elapsed 10.8s
Loss: 3.0498 | mIoU: 0.0164 | Acc: 0.2482
epoch 1, iter 0, loss: 3.1801
epoch 1, iter 20, loss: 3.0894
epoch 1, iter 40, loss: 2.7219
epoch 1, iter 60, loss: 2.3240
epoch 1, iter 80, loss: 2.6452
Finish epoch 1, time elapsed 11.0s
Loss: 2.7912 | mIoU: 0.0368 | Acc: 0.7330
epoch 2, iter 0, loss: 2.7508
epoch 2, iter 20, loss: 2.9132
epoch 2, iter 40, loss: 2.5194
epoch 2, iter 60, loss: 2.5761
epoch 2, iter 80, loss: 2.6568
Finish epoch 2, time elapsed 10.9s
Loss: 2.6078 | mIoU: 0.0377 | Acc: 0.7420
epoch 3, iter 0, loss: 2.5342
epoch 3, iter 20, loss: 2.6200
epoch 3, iter 40, loss: 2.7115
epoch 3, iter 60, loss: 2.6306
epoch 3, iter 80, loss: 2.3962
Finish epoch 3, time elapsed 10.7s
Loss: 2.5979 | mIoU: 0.0374 | Acc: 0.7240
epoch 4, iter 0, loss: 2.5300
epoc

In [ ]:
# import torch
# import torch.nn as nn
# import torch.nn.functional as F

# # --- Blocks ---

# def double_conv(in_ch, out_ch):
#     return nn.Sequential(
#         nn.Conv2d(in_ch, out_ch, 3, padding=1),
#         nn.ReLU(inplace=True),
#         nn.Conv2d(out_ch, out_ch, 3, padding=1),
#         nn.ReLU(inplace=True),
#     )

# class UNet(nn.Module):
#     def __init__(self, num_classes=21):
#         super().__init__()
#         # Encoder
#         self.e1 = double_conv(3,   64)
#         self.e2 = double_conv(64,  128)
#         self.e3 = double_conv(128, 256)
#         # Bottleneck
#         self.b  = double_conv(256, 512)
#         # Decoder
#         self.d3 = double_conv(512 + 256, 256)
#         self.d2 = double_conv(256 + 128, 128)
#         self.d1 = double_conv(128 + 64,   64)
#         # Output
#         self.out = nn.Conv2d(64, num_classes, 1)

#     def forward(self, x):
#         # Encode
#         s1 = self.e1(x)
#         s2 = self.e2(F.max_pool2d(s1, 2))
#         s3 = self.e3(F.max_pool2d(s2, 2))
#         # Bottleneck
#         x  = self.b(F.max_pool2d(s3, 2))
#         # Decode (upsample -> cat skip -> conv)
#         x  = self.d3(torch.cat([F.interpolate(x, s3.shape[2:]), s3], dim=1))
#         x  = self.d2(torch.cat([F.interpolate(x, s2.shape[2:]), s2], dim=1))
#         x  = self.d1(torch.cat([F.interpolate(x, s1.shape[2:]), s1], dim=1))
#         return self.out(x)

In [ ]:
# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# model     = UNet().to(device)
# optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
# criterion = nn.CrossEntropyLoss(ignore_index=255)

# for imgs, masks in train_loader:  # use train_loader, not train_dataset
#     imgs, masks = imgs.to(device), masks.to(device)
#     loss = criterion(model(imgs), masks)
#     optimizer.zero_grad()
#     loss.backward()
#     optimizer.step()